# 🍎 Apple Freshness Classifier: training notebook

This notebook trains a small vision model that labels an apple photo **fresh** or **rotten**. It then exports the model so the laptop app (`apple_checker.py`) can run it with no internet and no GPU.

**Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

What it does:
1. Downloads the public Kaggle dataset *Fruits fresh and rotten for classification* and keeps only the apples.
2. Adds **your own photos from the chamber**, if you've uploaded them to Google Drive. These matter most, because they look exactly like what the device will see.
3. Fine-tunes a pretrained **MobileNetV3** (transfer learning) in two stages.
4. Measures how often it would **wrongly call a rotten apple fresh**, and picks a safety threshold.
5. Exports `apple_model.onnx` + `model_info.json` to `MyDrive/apple_model/`.

Runs take about 10–15 minutes on a T4.

## 1. Setup

In [ ]:
!pip -q install timm kagglehub onnx onnxscript onnxruntime
import torch
print("PyTorch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE. Switch to Runtime > Change runtime type > T4 GPU, or training will be very slow")

## 2. Settings
Change these if you need to. The defaults are sensible.

In [ ]:
MODEL_NAME   = "mobilenetv3_large_100"  # fast on a laptop CPU. Alternatives: "efficientnet_b0", "resnet18"
PRETRAINED   = True
IMG_SIZE     = 224
BATCH_SIZE   = 32
HEAD_EPOCHS  = 3      # stage 1: train only the new classifier head
FT_EPOCHS    = 12     # stage 2: fine-tune the whole network
LR_HEAD      = 1e-3
LR_FT        = 2e-4

# Safety target: the "FRESH" verdict is only given when the model is confident enough that
# at least this share of rotten validation apples would NOT have been called fresh.
TARGET_ROTTEN_RECALL = 0.98

# ---- YOUR DATASET ----------------------------------------------------------
# Put your photos in two folders called "fresh" and "rotten", zip them, and upload the zip
# to Google Drive (My Drive). Change the name below if your zip is called something else.
MY_DATASET_ZIP = "/content/drive/MyDrive/apple_photos.zip"
OWN_PHOTOS_DIR = "/content/drive/MyDrive/apple_photos"  # used only if there's no zip (an unzipped folder in Drive)

USE_KAGGLE     = True   # download the public Kaggle fresh/rotten apples (if your dataset IS that set, no need to upload it)
KAGGLE_ROOT    = None   # None = download automatically
OWN_WEIGHT     = 5      # sample your own photos this many times more often than Kaggle ones
OUT_DIR        = "/content/drive/MyDrive/apple_model"
NUM_WORKERS    = 2
SEED           = 42
CLASSES        = ["fresh", "rotten"]   # index 1 = rotten

## 3. Connect Google Drive
Drive stores your own photos (`MyDrive/apple_photos`) and the trained model (`MyDrive/apple_model`). Allow access when it asks.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running on Colab: skipping Drive mount")

## 4. Load the data
Loads the Kaggle apples and/or your own zip, then cleans them up. Downloaded copies of this Kaggle dataset have two problems:
- **Exact duplicates:** every image in its `Test` folder also appears in `Train`. Duplicates are removed by comparing the file contents.
- **Augmented copies:** files such as `rotated_by_15_…`, `vertical_flip_…`, `translation_…` and `saltandpepper_…` are edited versions of the same original photo. All versions of one photo are kept together as a group.

The dataset's own Train/Test split is therefore **ignored**. Groups are re-split 70/15/15 instead, so the test set contains only photos the model has never seen in any form. The same goes for your own photos: every photo named `apple007_…` stays in the same split.

In [ ]:
import hashlib, random, collections, re, zipfile
from pathlib import Path
import numpy as np

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
AUG_PREFIX = re.compile(r"^(rotated_by_\d+_|translation_|saltandpepper_|vertical_flip_|horizontal_flip_)", re.I)

def md5(p):
    return hashlib.md5(Path(p).read_bytes()).hexdigest()

def label_from_folders(rel_path):
    # nearest folder whose name starts with "fresh" or "rotten" (freshapples, Fresh, rotten_apples ...)
    for part in reversed([x.lower() for x in rel_path.parts[:-1]]):
        if "banana" in part or "orange" in part:
            return None
        if part.startswith("fresh"):
            return 0
        if part.startswith("rotten"):
            return 1
    return None

def group_of(p, label):
    m = re.match(r"(apple\d+)_", p.name, re.I)
    if m:
        return m.group(1).lower()                           # apple007_003.jpg -> apple007
    return f"{label}/{AUG_PREFIX.sub('', p.stem).lower()}"  # rotated_by_15_X.png -> X

seen_hashes, report = set(), collections.Counter()
def load_folder(root, source):
    items = []
    for p in sorted(Path(root).rglob("*")):
        if p.suffix.lower() not in IMG_EXT or p.name.startswith(".") or "__MACOSX" in p.parts:
            continue
        label = label_from_folders(p.relative_to(root))
        if label is None:
            continue
        report[f"{source} files"] += 1
        h = md5(p)
        if h in seen_hashes:
            report[f"{source} exact duplicates removed"] += 1
            continue
        seen_hashes.add(h)
        items.append(dict(path=str(p), label=label, source=source, group=group_of(p, label)))
    return items

samples = []
if MY_DATASET_ZIP and Path(MY_DATASET_ZIP).exists():
    extract_to = Path("/content/my_dataset")
    if not extract_to.exists():
        print("Unzipping", MY_DATASET_ZIP, "...")
        zipfile.ZipFile(MY_DATASET_ZIP).extractall(extract_to)
    OWN_PHOTOS_DIR = str(extract_to)
if OWN_PHOTOS_DIR and Path(OWN_PHOTOS_DIR).exists():
    samples += load_folder(OWN_PHOTOS_DIR, "own")
else:
    print("No own photos found (checked", MY_DATASET_ZIP, "and", OWN_PHOTOS_DIR, ")")

if USE_KAGGLE:
    if KAGGLE_ROOT is None:
        import kagglehub
        KAGGLE_ROOT = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")
    samples += load_folder(KAGGLE_ROOT, "kaggle")

assert samples, "No images found. Folders must be named fresh... and rotten... (e.g. fresh/, rottenapples/)"

# split by group (original photo / apple), stratified by label: 70% train, 15% val, 15% test
groups = collections.defaultdict(list)
for s in samples:
    groups[(s["source"], s["group"])].append(s)
for label in (0, 1):
    keys = sorted(k for k, ss in groups.items() if ss[0]["label"] == label)
    random.shuffle(keys)
    n = len(keys); n_test = n_val = max(1, round(0.15 * n)) if n >= 3 else 0
    for i, k in enumerate(keys):
        for s in groups[k]:
            s["split"] = "test" if i < n_test else "val" if i < n_test + n_val else "train"

for k, v in report.items():
    print(f"{k}: {v}")
print(f"unique images: {len(samples)}  |  distinct original photos/apples: {len(groups)}")
counts = collections.Counter((s["split"], s["source"], CLASSES[s["label"]]) for s in samples)
for k in sorted(counts):
    print(f"  {k[0]:5s} {k[1]:6s} {k[2]:6s} {counts[k]:5d}")
for k in ("train", "val", "test"):
    assert any(s["split"] == k for s in samples), f"No {k} images: add more photos"

## 5. Look at some examples

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

show = random.sample([s for s in samples if s["split"] == "train"], k=min(12, len(samples)))
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, s in zip(axes.flat, show):
    ax.imshow(Image.open(s["path"]).convert("RGB")); ax.axis("off")
    ax.set_title(f'{CLASSES[s["label"]]} ({s["source"]})', color="green" if s["label"] == 0 else "red")
plt.tight_layout(); plt.show()

## 6. Datasets and augmentation
Augmentation randomly flips, rotates, crops and changes the brightness of images. This makes the model cope better with lighting and position changes in the chamber. Hue is shifted only a tiny bit, because colour is a real clue for rot.

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
RESIZE = int(round(IMG_SIZE * 256 / 224))

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(0.85, 1.15)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02),
    T.RandomApply([T.GaussianBlur(5)], p=0.2),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
# Must match the preprocessing in apple_checker.py
eval_tf = T.Compose([T.Resize(RESIZE), T.CenterCrop(IMG_SIZE), T.ToTensor(), T.Normalize(MEAN, STD)])

class AppleDS(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        s = self.items[i]
        return self.tf(Image.open(s["path"]).convert("RGB")), s["label"]

split = {k: [s for s in samples if s["split"] == k] for k in ("train", "val", "test")}

# Balance classes, and show your own chamber photos more often
cls_count = collections.Counter(s["label"] for s in split["train"])
weights = [(OWN_WEIGHT if s["source"] == "own" else 1.0) / cls_count[s["label"]] for s in split["train"]]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

dl = dict(
    train=DataLoader(AppleDS(split["train"], train_tf), BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS),
    val=DataLoader(AppleDS(split["val"], eval_tf), BATCH_SIZE, num_workers=NUM_WORKERS),
    test=DataLoader(AppleDS(split["test"], eval_tf), BATCH_SIZE, num_workers=NUM_WORKERS),
)
print({k: len(v) for k, v in split.items()})

## 7. Train (transfer learning)
**Stage 1** freezes the pretrained network and trains only the new 2-class head.
**Stage 2** unfreezes everything and fine-tunes at a lower learning rate.
The version with the lowest validation loss is kept.

In [ ]:
import copy, time, timm
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=len(CLASSES)).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

@torch.no_grad()
def predict(loader):
    model.eval(); probs, labels = [], []
    for x, y in loader:
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            out = model(x.to(device))
        probs.append(torch.softmax(out.float(), 1)[:, 1].cpu()); labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

def run_stage(epochs, lr, train_all):
    for p in model.parameters():
        p.requires_grad = train_all
    for p in model.get_classifier().parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(dl["train"]))
    for ep in range(epochs):
        model.train(); t0 = time.time(); tot = n = 0
        for x, y in dl["train"]:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=(device == "cuda")):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(y); n += len(y)
        p, yv = predict(dl["val"])
        eps = 1e-6
        vloss = float(-np.mean(yv * np.log(p + eps) + (1 - yv) * np.log(1 - p + eps)))
        vacc = float(np.mean((p >= 0.5) == yv))
        history.append((tot / n, vloss, vacc))
        flag = ""
        if vloss < best["loss"]:
            best.update(loss=vloss, state=copy.deepcopy(model.state_dict())); flag = "  <- best"
        print(f"{'full' if train_all else 'head'} ep {ep+1}/{epochs}  train_loss {tot/n:.3f}  "
              f"val_loss {vloss:.3f}  val_acc {vacc:.3f}  ({time.time()-t0:.0f}s){flag}")

history, best = [], dict(loss=float("inf"), state=None)
run_stage(HEAD_EPOCHS, LR_HEAD, train_all=False)
run_stage(FT_EPOCHS, LR_FT, train_all=True)
model.load_state_dict(best["state"])

h = np.array(history)
plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1); plt.plot(h[:, 0], label="train"); plt.plot(h[:, 1], label="val"); plt.title("loss"); plt.legend()
plt.subplot(1, 2, 2); plt.plot(h[:, 2]); plt.title("val accuracy"); plt.tight_layout(); plt.show()

## 8. Pick the safety threshold and evaluate
The worst mistake for this device is **calling a rotten apple fresh**. So the app uses three answers:
- **FRESH**: only if P(rotten) < `t_fresh`. This threshold is set on the validation set so that at least `TARGET_ROTTEN_RECALL` of rotten apples are *not* called fresh.
- **ROTTEN**: if P(rotten) ≥ `t_rotten`.
- **UNSURE, please check by hand**: anything in between.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

pv, yv = predict(dl["val"])
rot = np.sort(pv[yv == 1])
t_fresh = float(np.quantile(rot, 1 - TARGET_ROTTEN_RECALL, method="lower")) if len(rot) else 0.5
t_fresh = float(np.clip(t_fresh, 0.02, 0.5))
t_rotten = max(0.5, t_fresh)
print(f"t_fresh = {t_fresh:.3f}   t_rotten = {t_rotten:.3f}")

pt, yt = predict(dl["test"])
src = np.array([s["source"] for s in split["test"]])

def verdicts(p):
    return np.where(p >= t_rotten, "ROTTEN", np.where(p < t_fresh, "FRESH", "UNSURE"))

def report(mask, name):
    if mask.sum() == 0:
        return {}
    p, y, v = pt[mask], yt[mask], verdicts(pt[mask])
    r = dict(
        n=int(mask.sum()),
        accuracy_at_0_5=float(np.mean((p >= 0.5) == y)),
        rotten_called_fresh=int(np.sum((y == 1) & (v == "FRESH"))),
        fresh_called_rotten=int(np.sum((y == 0) & (v == "ROTTEN"))),
        unsure=int(np.sum(v == "UNSURE")),
    )
    print(f"\n== {name} test set ({r['n']} images) ==")
    print(f"accuracy @0.5: {r['accuracy_at_0_5']:.3f}")
    print(f"rotten apples wrongly called FRESH: {r['rotten_called_fresh']} / {int(np.sum(y == 1))}   <- the number that matters")
    print(f"fresh apples called ROTTEN:         {r['fresh_called_rotten']} / {int(np.sum(y == 0))}")
    print(f"UNSURE (sent to manual check):      {r['unsure']}")
    return r

metrics = {"all": report(np.ones_like(yt, bool), "ALL")}
for s_ in ("kaggle", "own"):
    if (src == s_).any():
        metrics[s_] = report(src == s_, s_.upper())

print()
print(classification_report(yt, (pt >= 0.5).astype(int), target_names=CLASSES, digits=3))
ConfusionMatrixDisplay(confusion_matrix(yt, (pt >= 0.5).astype(int)), display_labels=CLASSES).plot(cmap="Blues")
plt.title("Test confusion matrix (threshold 0.5)"); plt.show()

**Note:** these scores come from photos that were never seen in training, not even as a rotated copy. The Kaggle photos are still studio shots, so the real test is how the model does on photos taken inside **your chamber**. Once you have those, report that score too.

## 9. Look at the mistakes

In [ ]:
wrong = [i for i in range(len(yt)) if (pt[i] >= 0.5) != yt[i]]
print(len(wrong), "misclassified test images")
if wrong:
    k = min(12, len(wrong))
    fig, axes = plt.subplots(1, k, figsize=(2.5 * k, 3), squeeze=False)
    for ax, i in zip(axes[0], wrong[:k]):
        s = split["test"][i]
        ax.imshow(Image.open(s["path"]).convert("RGB")); ax.axis("off")
        ax.set_title(f'true {CLASSES[s["label"]]}\nP(rot)={pt[i]:.2f}', fontsize=9)
    plt.tight_layout(); plt.show()

## 10. Export for the laptop
Saves `apple_model.onnx` (with softmax built in) and `model_info.json` to `MyDrive/apple_model/`. The export is checked against PyTorch before it's saved.

In [ ]:
import json, os, onnxruntime as ort

os.makedirs(OUT_DIR, exist_ok=True)
onnx_path = os.path.join(OUT_DIR, "apple_model.onnx")

class WithSoftmax(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return torch.softmax(self.m(x), dim=1)

export_model = WithSoftmax(copy.deepcopy(model).float().cpu().eval()).eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
export_kw = dict(input_names=["image"], output_names=["probs"], opset_version=18,
                 dynamic_axes={"image": {0: "batch"}, "probs": {0: "batch"}})
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:
        torch.onnx.export(export_model, (dummy,), onnx_path, dynamo=False, **export_kw)
    except Exception as e:  # newer PyTorch without the classic exporter
        print("classic exporter failed, using dynamo exporter:", e)
        torch.onnx.export(export_model, (dummy,), onnx_path, dynamo=True, **export_kw)

# sanity check: ONNX Runtime must agree with PyTorch
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
x = torch.randn(4, 3, IMG_SIZE, IMG_SIZE)
diff = np.abs(sess.run(None, {"image": x.numpy()})[0] - export_model(x).detach().numpy()).max()
print("max ONNX vs PyTorch difference:", diff)
assert diff < 1e-3

info = dict(model_name=MODEL_NAME, classes=CLASSES, img_size=IMG_SIZE, resize=RESIZE, mean=MEAN, std=STD,
            t_fresh=t_fresh, t_rotten=t_rotten, target_rotten_recall=TARGET_ROTTEN_RECALL,
            test_metrics=metrics, trained_on=dict(collections.Counter(f'{s["source"]}/{CLASSES[s["label"]]}'
                                                                     for s in split["train"])))
with open(os.path.join(OUT_DIR, "model_info.json"), "w") as f:
    json.dump(info, f, indent=2)
torch.save(model.state_dict(), os.path.join(OUT_DIR, "apple_model.pt"))
print("Saved to", OUT_DIR, os.listdir(OUT_DIR))

In [ ]:
# Download the two files the laptop needs (they are also in Google Drive)
try:
    from google.colab import files
    files.download(onnx_path); files.download(os.path.join(OUT_DIR, "model_info.json"))
except ImportError:
    pass

## 11. Try it on any photo (optional)
Upload a photo of an apple and see what the model says.

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    for name in up:
        x = eval_tf(Image.open(name).convert("RGB")).unsqueeze(0).numpy()
        p = float(sess.run(None, {"image": x})[0][0, 1])
        v = "ROTTEN" if p >= t_rotten else "FRESH" if p < t_fresh else "UNSURE"
        plt.imshow(Image.open(name)); plt.axis("off"); plt.title(f"{v}   P(rotten)={p:.2f}"); plt.show()
except ImportError:
    pass

## Next steps
1. Put `apple_model.onnx` and `model_info.json` in the `model` folder on the laptop and run `run_checker.bat`.
2. Once the camera and chamber are ready, collect your own photos with `run_capture.bat`. Upload the `apple_photos` folder to **MyDrive/apple_photos** and run this notebook again. The **OWN** test score is the one to report.